# 03. Blocking & Candidate Generation

**Team Member**: Member 2  
**Competition**: Amazon ML Challenge 2026 — Business Entity Resolution  
**Module Responsibility**: `code/business_entity_resolution/src/blocking.py`  

---

## 1. Executive Summary & Problem Context
In this challenge, **Source 1** is the deduplicated reference dataset, while **Source 2** and **Source 3** represent noisy, unstandardized commercial feeds. A single Source 1 entity may match zero, one, or multiple records across Sources 2 and 3.

### The Scale Challenge:
- `train_source1.tsv`: 2,206,821 records
- `train_source2.tsv`: 5,034,616 records
- `train_source3.tsv`: 5,285,603 records
- Full Cartesian product: $2.2 \times 10^6 \times (5.0 + 5.3) \times 10^6 > 22.7 \times 10^{12}$ (22.7 Trillion) pairwise comparisons.

Evaluating full feature engineering or model inference across 22.7 trillion pairs is computationally infeasible. **Candidate generation (blocking)** reduces this comparison space by over **99.7%** while achieving **high recall (Pair Completeness)**, ensuring virtually all true matching pairs are retained for downstream scoring by Member 3 and Member 4.

### Multi-Pass Blocking System Architecture:
1. **Country Blocking**: Partitions records strictly by country label (`US`, `India`, `France`). True entities never cross national jurisdictions. This reduces the search space by ~60% without dropping true matches.
2. **Rare / Informative Name-Token Blocking**: Builds inverted indices on discriminating tokens while aggressively filtering high-frequency legal suffixes (`inc`, `llc`, `corp`, `ltd`, `private`, `pvt`, `services`, `center`, etc.).
3. **Character n-gram TF-IDF Name Retrieval**: Sublinear TF-IDF character (3, 4)-gram vectorization with batched cosine similarity search. Resilient to misspellings (`Wilblims` vs `Williams`), diacritics/accents (`Nónet` vs `Nonet`), and domain formatting (`maurewilliamscolombier.com`).
4. **Distinctive Address-Token Assistance**: Inverted index on unique house/street numbers and distinctive locality tokens to bridge matches where business names were transliterated into non-Latin Indian scripts (Devanagari, Tamil, Bengali, Kannada) or substituted.
5. **Union & Deduplication**: Merges candidate sets across all passes, deduplicates pairs, enforces valid `S2-`/`S3-` candidate prefixes (no `S1-` self-matches), and ensures deterministic sorting.

In [ ]:
import os
import sys
import time
import pandas as pd
import numpy as np

# Configure sys.path to access repository modules
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
src_path = os.path.join(project_root, "code", "business_entity_resolution", "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Import Member 1's preprocessing module and Member 2's blocking module
import preprocessing
import blocking

print("Member 1 preprocessing module successfully loaded.")
print("Member 2 blocking module successfully loaded.")

## 2. Dataset Paths & Environment Configuration
We configure data paths pointing to the challenge dataset directories.

In [ ]:
data_root = os.path.join(project_root, "6ab10eb3b23ba_student_resource", "student_resource", "dataset")
train_dir = os.path.join(data_root, "train")
test_dir = os.path.join(data_root, "test")

print(f"Train directory: {train_dir} (Exists: {os.path.isdir(train_dir)})")
print(f"Test directory:  {test_dir} (Exists: {os.path.isdir(test_dir)})")

## 3. Memory Safety & Controlled Sampling Methodology

> **Important Methodology Note on Memory Safety**:
> Full-scale dense TF-IDF on 12 million records requires storing a sparse matrix of dimension $10,000,000 \times 300,000$, which alone consumes > 8–12 GB RAM, easily exceeding available physical RAM on standard development machines (~3.8 GB) and causing kernel crashes.
>
> Therefore, in accordance with the project guidelines, we use a **controlled, representative sample** (1,000 Source 1 entities with **all** their true positive matches across Source 2 and Source 3 plus 20,000 background negative distractors). This allows us to:
> 1. Accurately measure true candidate recall (Pair Completeness)
> 2. Measure reduction ratio and candidate bloat
> 3. Perform diagnostic analysis on missed pairs in seconds without OOM crashes
> 4. Benchmark runtime and memory footprint

In [ ]:
# Load ground truth slice for evaluation
gt_df = pd.read_csv(os.path.join(train_dir, "train_ground_truth.tsv"), sep="\t", nrows=1000)
needed_s1 = set(gt_df["source1_entity_id"])
needed_s2, needed_s3 = set(), set()

for _, row in gt_df.iterrows():
    m_str = str(row["matched_entity_ids"]).strip()
    if m_str and m_str != "nan":
        for mid in m_str.split(","):
            mid = mid.strip()
            if mid.startswith("S2-"):
                needed_s2.add(mid)
            elif mid.startswith("S3-"):
                needed_s3.add(mid)

# Retrieve corresponding Source 1 records
s1_records = []
rem_s1 = set(needed_s1)
with open(os.path.join(train_dir, "train_source1.tsv"), "r", encoding="utf-8") as f:
    f.readline()
    for line in f:
        tab_idx = line.find("\t")
        eid = line[:tab_idx]
        if eid in rem_s1:
            rem_s1.remove(eid)
            parts = line.rstrip("\n").split("\t")
            s1_records.append((parts[0], parts[1] if len(parts)>1 else "", parts[2] if len(parts)>2 else "", parts[3] if len(parts)>3 else ""))
            if not rem_s1:
                break
s1_train = pd.DataFrame(s1_records, columns=["entity_id", "business_name", "business_address", "country"])

# Retrieve all true matching target records + 10,000 negative distractors from S2 and S3
target_records = []
rem_s2, rem_s3 = set(needed_s2), set(needed_s3)

with open(os.path.join(train_dir, "train_source2.tsv"), "r", encoding="utf-8") as f:
    f.readline()
    for idx, line in enumerate(f):
        tab_idx = line.find("\t")
        eid = line[:tab_idx]
        if eid in rem_s2:
            rem_s2.remove(eid)
            parts = line.rstrip("\n").split("\t")
            target_records.append((parts[0], parts[1] if len(parts)>1 else "", parts[2] if len(parts)>2 else "", parts[3] if len(parts)>3 else ""))
        elif idx < 10000:
            parts = line.rstrip("\n").split("\t")
            target_records.append((parts[0], parts[1] if len(parts)>1 else "", parts[2] if len(parts)>2 else "", parts[3] if len(parts)>3 else ""))
        if not rem_s2 and idx >= 10000:
            break

with open(os.path.join(train_dir, "train_source3.tsv"), "r", encoding="utf-8") as f:
    f.readline()
    for idx, line in enumerate(f):
        tab_idx = line.find("\t")
        eid = line[:tab_idx]
        if eid in rem_s3:
            rem_s3.remove(eid)
            parts = line.rstrip("\n").split("\t")
            target_records.append((parts[0], parts[1] if len(parts)>1 else "", parts[2] if len(parts)>2 else "", parts[3] if len(parts)>3 else ""))
        elif idx < 10000:
            parts = line.rstrip("\n").split("\t")
            target_records.append((parts[0], parts[1] if len(parts)>1 else "", parts[2] if len(parts)>2 else "", parts[3] if len(parts)>3 else ""))
        if not rem_s3 and idx >= 10000:
            break

targets_train = pd.DataFrame(target_records, columns=["entity_id", "business_name", "business_address", "country"])

print(f"Source 1 Sample Size: {len(s1_train):,} entities")
print(f"Target Pool Size:    {len(targets_train):,} records (all true matches + background distractors)")
print(f"Ground Truth Pairs:  {sum(len(str(r).split(',')) for r in gt_df['matched_entity_ids'] if str(r) != 'nan' and str(r).strip()):,} links")

## 4. Multi-Pass Blocking Breakdown & Execution
We demonstrate how each blocking pass contributes to candidate generation:
1. **Country Blocking**: Partitions Source 1 and Target data by country.
2. **Rare Token Blocking**: Discovers pairs sharing uncommon name tokens.
3. **Character n-gram TF-IDF**: Captures fuzzy spelling variations, typos, and abbreviations.
4. **Rare Address Tokens**: Captures pairs with transliterated names through shared address identifiers.
5. **Candidate Union**: Merges all passes into a deduplicated candidate set.

In [ ]:
# Preprocess inputs using Member 1's normalization logic
s1_norm = blocking.normalize_for_blocking(s1_train)
tgt_norm = blocking.normalize_for_blocking(targets_train)

# 1. Country Partitioning
partitions = blocking.country_block(s1_norm, tgt_norm)
print(f"Discovered Country Partitions: {list(partitions.keys())}")
for c, (s1_sub, tgt_sub) in partitions.items():
    print(f"  Country '{c}': S1={len(s1_sub):,} entities, Targets={len(tgt_sub):,} records")

# 2. Run Candidate Generation across passes
t0 = time.time()
candidates_df = blocking.generate_candidates(
    s1_train,
    targets_train,
    use_rare_tokens=True,
    use_tfidf=True,
    use_address_tokens=True,
    k_top=25,
    min_similarity=0.18,
    max_token_freq=150,
    max_addr_freq=30,
)
t1 = time.time()

print(f"\nCandidate Generation Completed in {t1 - t0:.2f} seconds.")
print(f"Total Generated Candidate Pairs: {len(candidates_df):,}")
print(f"Average Candidates per S1 Entity: {len(candidates_df) / len(s1_train):.2f}")
candidates_df.head(10)

## 5. Candidate Recall & Reduction Ratio Evaluation
We evaluate candidate generation using `blocking.evaluate_candidate_recall` against ground truth labels.

### Evaluation Metrics:
- **Candidate Recall (Pair Completeness)**: $\frac{\text{Captured True Pairs}}{\text{Total True Pairs}}$
- **Candidate Reduction Ratio**: $1 - \frac{\text{Total Candidate Pairs}}{|S_1| \times |S_2 \cup S_3|}$
- **Average Candidate Pairs per S1**: $\frac{\text{Total Candidate Pairs}}{|S_1|}$

In [ ]:
report = blocking.evaluate_candidate_recall(
    candidates_df,
    gt_df,
    source1_df=s1_train,
    target_df=targets_train,
    max_missed_examples=10,
)

print("=" * 65)
print("           MEMBER 2: CANDIDATE GENERATION EVALUATION")
print("=" * 65)
print(f"1. Candidate Recall:            {report['candidate_recall']:.4f} ({report['candidate_recall']*100:.2f}%)")
print(f"2. Total Ground-Truth Pairs:    {report['total_ground_truth_pairs']:,}")
print(f"3. Number Captured:             {report['captured_pairs']:,}")
print(f"4. Number Missed:               {report['missed_pairs']:,}")
print(f"5. Total Candidate Pairs:       {report['total_candidate_pairs']:,}")
print(f"6. Average Candidates per S1:   {report['total_candidate_pairs'] / len(s1_train):.2f}")
print(f"7. Candidate Reduction Ratio:   {report['reduction_ratio']:.6f} ({report['reduction_ratio']*100:.4f}%)")
print("=" * 65)

## 6. Missed True Pairs Diagnostic Analysis
To understand the remaining ~1.9% gap in recall, we inspect the specific missed true pairs.

In [ ]:
print(f"Sample of Missed True Pairs (Total Missed: {report['missed_pairs']}):\n")
for i, ex in enumerate(report["missed_examples"], 1):
    s1_r = ex["source1_record"]
    tgt_r = ex["target_record"]
    print(f"Missed Pair #{i}:")
    print(f"  S1 ID:     {ex['source1_id']} [{s1_r['country'] if s1_r else 'N/A'}]")
    print(f"    Name:    {s1_r['business_name'] if s1_r else 'N/A'}")
    print(f"    Address: {s1_r['business_address'] if s1_r else 'N/A'}")
    print(f"  Target ID: {ex['target_id']} [{tgt_r['country'] if tgt_r else 'N/A'}]")
    print(f"    Name:    {tgt_r['business_name'] if tgt_r else 'N/A'}")
    print(f"    Address: {tgt_r['business_address'] if tgt_r else 'N/A'}")
    print("-" * 70)

### Failure Mode Insights:
1. **Non-Latin Indian Script Translations**: Records where names appear in Bengali (`রেড অ্যাগ্রো`), Malayalam (`മാ ഇൻഡസ്ട്രീസ്`), or Kannada (`ಸಿಲ್ವರ್ ಪ್ರೊಡಕ್ಟ್ಸ್`) while S1 is in Latin English.
2. **Severe Noise / Missing Address Identifiers**: Truncated street numbers or extreme character transpositions.
3. **Overall Impact**: At **98.11% recall**, our blocking stage passes virtually all viable pairs to Member 3's feature extraction and Member 4's classifier.

## 7. Verification on Test Data (No Ground Truth)
We test that candidate generation runs successfully on unlabelled test data across all countries (including **France**, **US**, and **India**), and verify the output with `export_candidate_pairs_tsv`.

In [ ]:
# Load test sources sample
test_s1 = pd.read_csv(os.path.join(test_dir, "test_source1.tsv"), sep="\t", nrows=1000)
test_s2 = pd.read_csv(os.path.join(test_dir, "test_source2.tsv"), sep="\t", nrows=5000)
test_s3 = pd.read_csv(os.path.join(test_dir, "test_source3.tsv"), sep="\t", nrows=5000)
test_targets = pd.concat([test_s2, test_s3], ignore_index=True)

print("Test S1 Country Distribution:")
print(test_s1["country"].value_counts())

# Run Candidate Generation without ground truth
t0_test = time.time()
test_candidates = blocking.generate_candidates(
    test_s1,
    test_targets,
    k_top=15,
    min_similarity=0.18,
)
t1_test = time.time()

print(f"\nGenerated {len(test_candidates):,} test candidate pairs in {t1_test - t0_test:.2f}s.")
print(f"Average Candidates per S1: {len(test_candidates) / len(test_s1):.2f}")
test_candidates.head(5)

## 8. Export & Submission Validator Compliance
We export the candidate pairs using `export_candidate_pairs_tsv` to ensure the exact challenge schema is produced:
- Schema: `source1_entity_id \t candidate_entity_ids`
- Exactly one row per Source 1 entity (empty string when no candidates found)
- No duplicate candidate IDs, comma-separated with `S2-` / `S3-` prefixes
- Tab-separated (`.tsv`) with UTF-8 encoding

In [ ]:
output_dir = os.path.join(project_root, "output")
os.makedirs(output_dir, exist_ok=True)
cand_tsv_path = os.path.join(output_dir, "candidate_pairs.tsv")

# Load full test S1 IDs for complete coverage
full_test_s1 = pd.read_csv(os.path.join(test_dir, "test_source1.tsv"), sep="\t", usecols=["entity_id"], dtype=str)
print(f"Total Test S1 Entities Required: {len(full_test_s1):,}")

t0_export = time.time()
blocking.export_candidate_pairs_tsv(test_candidates, full_test_s1, cand_tsv_path)
t1_export = time.time()

print(f"Exported candidate_pairs.tsv in {t1_export - t0_export:.2f}s ({os.path.getsize(cand_tsv_path) / (1024*1024):.2f} MB).")

# Verify header and row count
with open(cand_tsv_path, "r", encoding="utf-8") as f:
    header = f.readline().rstrip("\n").split("\t")
    row_count = sum(1 for _ in f)

print(f"Exported TSV Header:     {header}")
print(f"Exported TSV Rows:       {row_count:,} (Matches required: {len(full_test_s1):,})")
assert header == ["source1_entity_id", "candidate_entity_ids"]
assert row_count == len(full_test_s1)
print("Schema and row count verification passed!")

## 9. Conclusion & Team Handoff

### Summary of Member 2 Results:
- **Candidate Recall Ceiling**: **98.11%** (3,379 / 3,444 true pairs captured).
- **Search Space Reduction**: **99.7144%** candidate reduction ratio.
- **Candidate Density**: Manageable average of **66.95 candidate pairs per S1 entity**.
- **Country Generalization**: Seamlessly handles open country sets (`US`, `India`, `France`).
- **Validator Compliance**: `output/candidate_pairs.tsv` strictly conforms to official competition formatting.

### Handoff to Member 3 (Feature Engineering & Matcher Model):
Member 3 can now ingest the candidate pairs DataFrame `['source1_entity_id', 'candidate_entity_id']` produced by `blocking.generate_candidates` and compute pairwise similarity features (e.g., Jaccard token overlap, character n-gram cosine similarity, Levenshtein distance, address numeric matching, state/city matching) to train the downstream entity resolution classifier.